
# Swiggy Instamart Customer Intelligence Support Co-Pilot

**Prepared by:** D.Kaviarasu  
**Course:** IIT Patna – AI & ML Course Project

## Project Overview

This project demonstrates a customer intelligence and support system using synthetic Instamart-style data.

### Main components
1. Data collection, cleaning and exploration
2. SQLite database and SQL analysis
3. RFM customer segmentation using KMeans
4. PCA dimensionality reduction
5. Customer churn prediction
6. Support ticket classification and team routing
7. Policy retrieval and template-based customer support
8. Integrated customer intelligence co-pilot

**Important:** The datasets and policy documents are simulated for academic demonstration. They are not actual Swiggy customer records or official Swiggy policies. The response generator is template-based, not a live LLM.



## 1. Project Setup and Libraries

Load the project modules and prepare the working directory.


In [1]:

import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/content/instamart-copilot-workspace")
sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)
print("Directory exists:", PROJECT_DIR.exists())


Project directory: /content/instamart-copilot-workspace
Directory exists: True



## 2. Load the Customer, Order and Ticket Data

The project uses generated synthetic datasets for demonstration.


In [2]:

customers = pd.read_csv(PROJECT_DIR / "data/cleaned/customers_cleaned.csv")
orders = pd.read_csv(PROJECT_DIR / "data/cleaned/orders_cleaned.csv")
tickets = pd.read_csv(PROJECT_DIR / "data/cleaned/tickets_cleaned.csv")

print("CUSTOMERS:", customers.shape)
display(customers.head())

print("ORDERS:", orders.shape)
display(orders.head())

print("TICKETS:", tickets.shape)
display(tickets.head())


CUSTOMERS: (1000, 5)


,customer_id,city,gender,age,signup_date
0,CUST0001,Delhi,Other,37.0,2026-02-25
1,CUST0002,Hyderabad,Female,39.0,2026-06-06
2,CUST0003,Mumbai,Other,28.0,2026-02-08
3,CUST0004,Delhi,Male,44.0,2023-05-16
4,CUST0005,Bangalore,Male,20.0,2023-12-28


ORDERS: (5000, 6)


,order_id,customer_id,order_date,order_amount,order_status,payment_method
0,ORD00001,CUST0455,2025-10-08,2025.02,Delivered,Wallet
1,ORD00002,CUST0013,2025-08-03,1520.50,Cancelled,Upi
2,ORD00003,CUST0210,2026-09-02,1600.55,Delivered,Upi
3,ORD00004,CUST0741,2026-02-24,1353.83,Delivered,Card
4,ORD00005,CUST0397,2025-04-11,2975.93,Delivered,Wallet


TICKETS: (1000, 8)


,ticket_id,customer_id,order_id,issue_category,description,priority,ticket_status,created_at
0,TKT00001,CUST0368,ORD00447,Refund Request,I want a refund for my order.,Low,In Progress,2025-03-14
1,TKT00002,CUST0115,ORD03870,Delivery Delay,My order has not arrived on time.,High,In Progress,2026-02-03
2,TKT00003,CUST0229,ORD03273,Product Quality,The product received was damaged or poor quality.,Low,Open,2025-04-05
3,TKT00004,CUST0112,ORD02195,Payment Issue,My payment was deducted but order confirmation...,Low,Resolved,2026-04-20
4,TKT00005,CUST0899,ORD02166,Delivery Delay,My order has not arrived on time.,High,In Progress,2025-04-05



## 3. Data Quality Checks

Check missing values and duplicate records in the cleaned datasets.


In [3]:

for name, df in [
    ("Customers", customers),
    ("Orders", orders),
    ("Tickets", tickets)
]:
    print("\\n", name)
    print("Rows and columns:", df.shape)
    print("Total missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())


\n Customers
Rows and columns: (1000, 5)
Total missing values: 0
Duplicate rows: 0
\n Orders
Rows and columns: (5000, 6)
Total missing values: 0
Duplicate rows: 0
\n Tickets
Rows and columns: (1000, 8)
Total missing values: 0
Duplicate rows: 0



## 4. Exploratory Data Analysis

Explore order status, customer distribution and support ticket categories.


In [4]:

print("CUSTOMERS BY CITY")
display(customers["city"].value_counts())

print("ORDERS BY STATUS")
display(orders["order_status"].value_counts())

print("TICKETS BY CATEGORY")
display(tickets["issue_category"].value_counts())


CUSTOMERS BY CITY


,count
city,
Hyderabad,156
Coimbatore,156
Mumbai,148
Delhi,138
Chennai,137
Pune,135
Bangalore,130


ORDERS BY STATUS


,count
order_status,
Delivered,4396
Cancelled,412
Returned,192


TICKETS BY CATEGORY


,count
issue_category,
Delivery Delay,226
Refund Request,220
Missing Item,215
Payment Issue,170
Product Quality,169



## 5. RFM Customer Segmentation

RFM stands for:

- **Recency:** How recently a customer made a delivered order.
- **Frequency:** Number of delivered orders.
- **Monetary:** Total spending on delivered orders.

The following files contain the prepared RFM metrics and KMeans segment assignments.


In [5]:

rfm = pd.read_csv(PROJECT_DIR / "data/cleaned/rfm_data.csv")
rfm_segmented = pd.read_csv(PROJECT_DIR / "data/cleaned/rfm_segmented.csv")

print("RFM DATA")
display(rfm.head())

print("CLUSTER COUNTS")
display(rfm_segmented["Cluster"].value_counts().sort_index())

print("CLUSTER PROFILES")
display(
    rfm_segmented.groupby("Cluster")[
        ["Recency", "Frequency", "Monetary"]
    ].mean().round(2)
)


RFM DATA


,customer_id,Recency,Frequency,Monetary
0,CUST0001,69,6,9229.43
1,CUST0002,87,3,3798.16
2,CUST0003,326,4,4582.77
3,CUST0004,440,2,5079.48
4,CUST0005,24,5,5660.40


CLUSTER COUNTS


,count
Cluster,
0,376
1,607


CLUSTER PROFILES


,Recency,Frequency,Monetary
Cluster,,,
0,237.47,2.52,3550.22
1,90.55,5.68,9055.34



## 6. PCA – Dimensionality Reduction

Principal Component Analysis reduces the three RFM variables into two principal components.

The following cell calculates the explained variance using standardized RFM features.


In [6]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

rfm_features = rfm[["Recency", "Frequency", "Monetary"]]

scaled_rfm = StandardScaler().fit_transform(rfm_features)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_rfm)

print("Explained variance by component:")
print(pca.explained_variance_ratio_)

print("Total variance explained:")
print(round(pca.explained_variance_ratio_.sum(), 4))


Explained variance by component:
[0.72881609 0.23301159]
Total variance explained:
0.9618



## 7. Customer Churn Prediction

The trained Logistic Regression pipeline is loaded from the models directory.

**Interpretation:** The project uses a synthetic churn label based on a recency rule. Its predictions are demonstrations using simulated data, not validated real-world churn probabilities.


In [7]:

import joblib

churn_model_path = PROJECT_DIR / "models/churn_model_improved.pkl"
churn_model = joblib.load(churn_model_path)

sample_customer = pd.DataFrame({
    "Frequency": [3],
    "Monetary": [4000]
})

prediction = churn_model.predict(sample_customer)[0]
probabilities = churn_model.predict_proba(sample_customer)[0]

print("Churn prediction:", int(prediction))
print("Churn probability:", round(probabilities[1] * 100, 2), "%")
print("Non-churn probability:", round(probabilities[0] * 100, 2), "%")


Churn prediction: 1
Churn probability: 62.74 %
Non-churn probability: 37.26 %



## 8. Support Ticket Classification and Routing

A TF-IDF and Logistic Regression model classifies customer complaints. A routing dictionary assigns the ticket to a support team.


In [8]:

from src.triage import process_ticket

sample_complaint = (
    "My payment was deducted but order confirmation failed."
)

ticket_result = process_ticket(
    description=sample_complaint,
    priority="High",
    status="Open"
)

print("TICKET CLASSIFICATION")
for key, value in ticket_result.items():
    print(f"{key}: {value}")


TICKET CLASSIFICATION
description: My payment was deducted but order confirmation failed.
category: Payment Issue
priority: High
team: Payment Support
status: Open



## 9. Policy Retrieval and Customer Support

The system searches mock policy documents using TF-IDF similarity and generates a template-based response from the retrieved content.


In [9]:

from src.policy_rag import generate_answer

customer_question = (
    "My payment was deducted but my order was not confirmed."
)

support_result = generate_answer(customer_question)

print("CUSTOMER QUESTION:")
print(customer_question)

print("\\nASSISTANT RESPONSE:")
print(support_result["answer"])

print("\\nPOLICY SOURCES:")
print(support_result["sources"])

print("Escalation required:", support_result["escalation"])


CUSTOMER QUESTION:
My payment was deducted but my order was not confirmed.
\nASSISTANT RESPONSE:
Based on the simulated project policy:

MOCK INSTAMART PROJECT POLICY: PAYMENT ISSUE
This is a simulated policy created for the academic project.
1. Customers should report payments deducted without order confirmation.
4. Payment-related refunds must be verified before approval.
5. Unresolved payment disputes should be escalated to Payment Support or a human agent.
2. The support team should verify the payment and order status.
3. Duplicate or failed transactions should be investigated.

Recommended next step:
The support team should review the relevant details and follow the applicable policy. Unresolved cases should be escalated to a human support agent.
\nPOLICY SOURCES:
['payment_issue.txt']
Escalation required: False



## 10. Integrated Customer Intelligence Support Co-Pilot

The final application combines churn prediction, ticket triage and policy-based customer support.


In [10]:

from src.copilot import run_copilot

final_result = run_copilot(
    frequency=3,
    monetary=4000,
    complaint="My payment was deducted but order confirmation failed.",
    question="My payment was deducted but my order was not confirmed."
)



SWIGGY INSTAMART CUSTOMER INTELLIGENCE

1. CUSTOMER CHURN ANALYSIS
Prediction: 1
Churn probability: 62.74 %

2. SUPPORT TICKET TRIAGE
Category: Payment Issue
Priority: Medium
Assigned team: Payment Support
Status: Open

3. POLICY-BASED CUSTOMER SUPPORT
Customer question: My payment was deducted but my order was not confirmed.

Assistant response:
Based on the simulated project policy:

MOCK INSTAMART PROJECT POLICY: PAYMENT ISSUE
This is a simulated policy created for the academic project.
1. Customers should report payments deducted without order confirmation.
4. Payment-related refunds must be verified before approval.
5. Unresolved payment disputes should be escalated to Payment Support or a human agent.
2. The support team should verify the payment and order status.
3. Duplicate or failed transactions should be investigated.

Recommended next step:
The support team should review the relevant details and follow the applicable policy. Unresolved cases should be escalated to a human 


## 11. Project Conclusion

The project demonstrates how machine learning, customer segmentation, text classification and policy retrieval can be combined into a customer support co-pilot.

### Completed components
- Synthetic data generation and cleaning
- Customer, order and ticket analysis
- SQLite database creation and SQL analysis
- RFM metrics and KMeans segmentation
- PCA variance analysis
- Logistic Regression churn prediction
- TF-IDF ticket classification and team routing
- Policy document retrieval
- Integrated customer intelligence application

### Limitations
- All customer and transaction data are synthetic.
- Churn labels are generated using a simplified recency-based rule.
- Ticket descriptions are simulated and repetitive, so test accuracy may not represent real-world performance.
- Policy responses are generated using a template-based mock response function, not a live language model.
- The policy documents are simulated and are not official Swiggy policies.

**Prepared by D.Kaviarasu**  
**IIT Patna – AI & ML Course Project**
